# Directory → CollectionType & Album → Collection migration

Tracker rows **#6 (Directory)** and **#14 (Album)** — Collection half only.

Scope decisions (2026-08-06):
- **Designer Tours** directory + its 59 albums are **skipped** — they become
  dx-card / Destination Expert data later (tracker rows #11/#13).
- **Journey subcollections are NOT created here** (deferred). The dual-price
  fix (`subcollections.price_starting_at`, migration
  `20260806060000_add_subcollection_price_starting_at`) is already in the
  schema for when that step runs.

Prerequisites: geo, media, companies, users migrated; `python scripts/seed.py`
run so the base collection types exist.

Run cells top to bottom. Idempotent — safe to re-run.

In [2]:
import os, re, json
from pathlib import Path

import requests
import psycopg
from psycopg.types.json import Json
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]

# legacy->new user id map from the user migration — pick the file matching THIS DB
USER_MAP_FILE = os.environ["USER_MAP_FILE"]


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def rel_many(obj):
    """Unwrap a populated to-many relation into a list of flat dicts."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    return [attrs(x) for x in (obj or [])]


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, "sort": "id", **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=120)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

connected to: production


## 1. Directory → `collection_types` (tracker #6)

The v1 directory list is reconciled to the fixed v2 set. Fixed v2 names/slugs
are kept; legacy `description` and `logo` URL are carried into
`description`/`icon`. Legacy `label`, `seo` and timestamps are dropped.
Designer Tours is skipped (dx-card migration will handle Destination Expert).

In [3]:
conn.rollback()  # clear any aborted transaction from a previous failed run

# legacy directory slug -> (name, slug, has_dedicated_collection, priority)
DIRECTORY_TO_CT = {
    "mindful-luxury-hotels": ("Properties",  "properties",  True,  1),  # Postcard StarPartner Stays
    "food-and-beverages":    ("Restaurants", "restaurants", False, 2),  # Food and Beverages
    "postcard-events":       ("Events",      "events",      False, 3),  # Postcard Events
    "postcard-shopping":     ("Shopping",    "shopping",    False, 4),  # Postcard Shopping
}
SKIP_DIRECTORY_SLUGS = {"mindful-luxury-tours"}  # Designer Tours -> dx-card migration later

directories = fetch_all("/api/directories", {"populate": "logo"})
print(f"fetched {len(directories)} directories")

dir_id_to_ct_slug, SKIPPED_DIR_IDS, unmapped_dirs = {}, set(), []
with conn.cursor() as cur:
    for d in directories:
        a = attrs(d)
        if a.get("slug") in SKIP_DIRECTORY_SLUGS:
            SKIPPED_DIR_IDS.add(d["id"])
            continue
        target = DIRECTORY_TO_CT.get(a.get("slug"))
        if not target:
            unmapped_dirs.append((d["id"], a.get("name"), a.get("slug")))
            continue
        name, slug, dedicated, priority = target

        logo, icon = rel(a.get("logo")), None
        if logo and logo.get("url"):
            icon = logo["url"].strip()
            if icon.startswith("/"):
                icon = CMS_BASE_URL + icon

        cur.execute(
            """
            INSERT INTO collection_types
                (name, slug, description, icon, has_dedicated_collection, priority)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (slug) DO UPDATE
            SET name = EXCLUDED.name,
                description = EXCLUDED.description,
                icon = EXCLUDED.icon,
                has_dedicated_collection = EXCLUDED.has_dedicated_collection,
                priority = EXCLUDED.priority
            """,
            (name, slug, (a.get("description") or "").strip() or None, icon, dedicated, priority),
        )
        dir_id_to_ct_slug[d["id"]] = slug
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT slug, id FROM collection_types")
    ct_id_by_slug = dict(cur.fetchall())

DIR_ID_TO_CT = {did: ct_id_by_slug[s] for did, s in dir_id_to_ct_slug.items()}
DEFAULT_CT_ID = ct_id_by_slug["properties"]  # fallback for albums with no directory

print("collection_types         :", ct_id_by_slug)
print("directory -> ct id       :", DIR_ID_TO_CT)
print("skipped directory ids    :", SKIPPED_DIR_IDS)   # Designer Tours
print("MANUAL REVIEW (unmapped) :", unmapped_dirs)     # should be empty

fetched 5 directories
collection_types         : {'properties': 1, 'restaurants': 2, 'events': 3, 'shopping': 4}
directory -> ct id       : {2: 1, 6: 2, 7: 3, 8: 4}
skipped directory ids    : {1}
MANUAL REVIEW (unmapped) : []


## 2. Fetch all albums (~29 paginated requests, expect 2827)

In [4]:
albums = sorted(fetch_all("/api/albums", {"populate": "*"}), key=lambda x: x["id"])
print(f"fetched {len(albums)} albums")

fetched 2826 albums


## 3. DB lookup maps + media find-or-create helper

Countries/regions/localities/companies were migrated by name, so albums are
matched the same way. Media is keyed by normalized URL (same normalization as
`scripts/media.py`, so existing rows are reused, never duplicated).

In [5]:
conn.rollback()

with conn.cursor() as cur:
    cur.execute("SELECT LOWER(name), id FROM countries")
    country_by_name = dict(cur.fetchall())
    cur.execute("SELECT LOWER(name), country_id, id FROM regions")
    region_by_name_country = {(n, c): i for n, c, i in cur.fetchall()}
    cur.execute("SELECT LOWER(name), id FROM localities")
    locality_by_name = {}
    for n, i in cur.fetchall():
        locality_by_name.setdefault(n, []).append(i)
    cur.execute("SELECT LOWER(name), id FROM companies")
    company_by_name = dict(cur.fetchall())
    cur.execute("SELECT slug, id FROM companies")
    company_by_slug = dict(cur.fetchall())
    cur.execute("SELECT url, id FROM media")
    media_by_url = dict(cur.fetchall())

print(f"lookups: {len(country_by_name)} countries, {len(region_by_name_country)} regions, "
      f"{len(locality_by_name)} locality names, {len(company_by_name)} companies, "
      f"{len(media_by_url)} media")


def media_id_for(image, cur):
    """Find-or-create a media row for a populated Strapi file."""
    if not image or not image.get("url"):
        return None
    url = image["url"].strip()
    if url.startswith("/"):
        url = CMS_BASE_URL + url
    if url in media_by_url:
        return media_by_url[url]
    cur.execute(
        "INSERT INTO media (url, mime_type, alt, width, height) VALUES (%s, %s, %s, %s, %s) RETURNING id",
        (url, image.get("mime"), image.get("alternativeText") or image.get("name"),
         image.get("width"), image.get("height")),
    )
    media_by_url[url] = cur.fetchone()[0]
    return media_by_url[url]

lookups: 264 countries, 1181 regions, 286 locality names, 207 companies, 15693 media


## 4. Album → `collections` (tracker #14)

- Designer Tours albums are skipped entirely.
- 665 legacy albums have an empty slug → generated from name, de-duplicated
  in-run (id-sorted, so `foo-2` suffixes stay stable across re-runs).
- `status`: legacy values match the v2 enum; nulls fall back to
  `isActive` → live/draft.
- Dropped fields (reviewed 2026-08-06): `signature` (copy of name),
  `on_boarding`, `news_article`, timestamps, `fixedDates`/`placeId`/`locationLink`
  (empty everywhere); `album_themes`/`category`/`environment`/`cuisines` →
  facet migrations; `follow_albums` → blocked Circle work; `postcards`/`memories`
  → their own tracker rows; journey/price fields → deferred Journey split.

In [6]:
conn.rollback()

VALID_STATUS = {"draft", "assigned", "submit", "rework", "live"}

used_slugs = set()
def unique_slug(base):
    base = base or "album"
    slug, n = base, 2
    while slug in used_slugs:
        slug = f"{base}-{n}"
        n += 1
    used_slugs.add(slug)
    return slug

album_to_collection = {}       # legacy album id -> new collection id
collection_slug_by_album = {}

skipped_no_name, skipped_designer_tours, no_directory = [], [], []
missing_country, missing_region, ambiguous_locality, unmatched_company = [], [], [], []

with conn.cursor() as cur:
    for al in albums:
        a = attrs(al)
        name = (a.get("name") or "").strip()
        if not name:
            skipped_no_name.append(al["id"])
            continue

        # directory -> collection_type; Designer Tours albums are NOT migrated
        dirs = rel_many(a.get("directories"))
        if dirs and dirs[0]["id"] in SKIPPED_DIR_IDS:
            skipped_designer_tours.append((al["id"], name))
            continue
        ct_id = DIR_ID_TO_CT.get(dirs[0]["id"], DEFAULT_CT_ID) if dirs else DEFAULT_CT_ID
        if not dirs:
            no_directory.append((al["id"], name))

        slug = unique_slug((a.get("slug") or "").strip() or slugify(name))

        # geo: country by name, region by (name, country), locality by unique name
        country = rel(a.get("country"))
        country_id = country_by_name.get((country.get("name") or "").strip().lower()) if country else None
        if country and not country_id:
            missing_country.append((al["id"], country.get("name")))

        region, region_id = rel(a.get("region")), None
        if region and country_id:
            region_id = region_by_name_country.get(((region.get("name") or "").strip().lower(), country_id))
        if region and not region_id:
            missing_region.append((al["id"], region.get("name")))

        locality, locality_id = rel(a.get("locality")), None
        if locality:
            ids = locality_by_name.get((locality.get("name") or "").strip().lower(), [])
            if len(ids) == 1:
                locality_id = ids[0]
            else:
                ambiguous_locality.append((al["id"], locality.get("name"), len(ids)))

        # company relation by name, else legacy companySlug string by slug
        company, company_id = rel(a.get("company")), None
        if company:
            company_id = company_by_name.get((company.get("name") or "").strip().lower())
            if not company_id:
                unmatched_company.append((al["id"], company.get("name")))
        elif (a.get("companySlug") or "").strip():
            cs = a["companySlug"].strip()
            company_id = company_by_slug.get(cs) or company_by_slug.get(slugify(cs))
            if not company_id:
                unmatched_company.append((al["id"], cs))

        cover_id = media_id_for(rel(a.get("coverImage")), cur)

        location = {k: v for k, v in {
            "lat": a.get("lat"), "lng": a.get("long"),
            "google_place_id": a.get("placeId"), "location_link": a.get("locationLink"),
        }.items() if v not in (None, "")} or None

        seo = {k: v for k, v in (a.get("seo") or {}).items()
               if k != "id" and v not in (None, "")} or None

        status = a.get("status") if a.get("status") in VALID_STATUS \
            else ("live" if a.get("isActive") else "draft")

        cur.execute(
            """
            INSERT INTO collections
                (collection_type_id, name, intro, story, slug, cover_media_id, seo,
                 is_featured, priority, country_id, region_id, locality_id, location,
                 managed_by_company_id, website, media_kit, additional_info,
                 sustainability, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (slug) DO UPDATE
            SET collection_type_id = EXCLUDED.collection_type_id,
                name = EXCLUDED.name,
                intro = EXCLUDED.intro,
                story = EXCLUDED.story,
                cover_media_id = EXCLUDED.cover_media_id,
                seo = EXCLUDED.seo,
                is_featured = EXCLUDED.is_featured,
                priority = EXCLUDED.priority,
                country_id = EXCLUDED.country_id,
                region_id = EXCLUDED.region_id,
                locality_id = EXCLUDED.locality_id,
                location = EXCLUDED.location,
                managed_by_company_id = EXCLUDED.managed_by_company_id,
                website = EXCLUDED.website,
                media_kit = EXCLUDED.media_kit,
                additional_info = EXCLUDED.additional_info,
                sustainability = EXCLUDED.sustainability,
                status = EXCLUDED.status
            RETURNING id
            """,
            (ct_id, name,
             (a.get("intro") or "").strip() or None,
             (a.get("story") or "").strip() or None,
             slug, cover_id, Json(seo) if seo else None,
             bool(a.get("isFeatured")), a.get("priority") or 0,
             country_id, region_id, locality_id,
             Json(location) if location else None,
             company_id,
             (a.get("website") or "").strip() or None,
             (a.get("media_kit") or "").strip() or None,
             (a.get("additionalInfo") or "").strip() or None,
             (a.get("sustainability") or "").strip() or None,
             status),
        )
        album_to_collection[al["id"]] = cur.fetchone()[0]
        collection_slug_by_album[al["id"]] = slug

conn.commit()
print(f"collections upserted: {len(album_to_collection)}")
print(f"skipped Designer Tours albums ({len(skipped_designer_tours)})")  # expect 59
print(f"skipped (no name): {skipped_no_name}")
print(f"no directory -> defaulted to Properties ({len(no_directory)}): {no_directory}")
print(f"MANUAL REVIEW country not found ({len(missing_country)}): {missing_country[:20]}")
print(f"MANUAL REVIEW region not found ({len(missing_region)}): {missing_region[:20]}")
print(f"MANUAL REVIEW locality missing/ambiguous ({len(ambiguous_locality)}): {ambiguous_locality[:20]}")
print(f"MANUAL REVIEW company unmatched ({len(unmatched_company)}): {unmatched_company[:20]}")

collections upserted: 2767
skipped Designer Tours albums (59)
skipped (no name): []
no directory -> defaulted to Properties (35): [(26, 'Rewilding Jaguars in Argentina'), (31, 'Baja Island Hopping, Mexico'), (32, 'In the footsteps of the Alchemist'), (33, 'The Mindful Triangle of India'), (34, 'Off beaten path trip to Lake Turkana, Lake Paradise and Chalbi Desert'), (46, 'Captivating Cuba - A destination unlike any other'), (48, 'P1 AYNI : Reciprocity, solidarity and humanity in all'), (49, 'P2 AYNI : Reciprocity, solidarity and humanity in all.'), (52, 'Connecting with Cuba and its culture, people and places'), (64, 'Travel for Impact in Uganda'), (224, 'Arctic Retreat AB'), (261, 'Jayu'), (295, 'Bandra_Kurla_Complex'), (321, 'The Spirit of the Jungle The Spirit of the JungleThe Spirit of the JungleThe Spirit of the Jungle'), (322, 'The Spirit of the Jungle The Spirit of the JungleThe Spirit of the JungleThe Spirit of the Jungle'), (324, 'The Spirit of t'), (328, 'African Adventure'),

## 5. Save the legacy album id map

`legacy_album_id_map.json` (legacy album id → new collection id) — the
postcards migration needs it. Rename per environment like the user maps.

In [7]:
out = ROOT / "legacy_album_id_map.json"
out.write_text(json.dumps({str(k): str(v) for k, v in album_to_collection.items()}, indent=2))
print(f"saved {len(album_to_collection)} legacy->new album id mappings to {out}")

saved 2767 legacy->new album id mappings to c:\Users\ReTechie\Desktop\postcard\postcard-migration\legacy_album_id_map.json


## 6. OPTIONAL — author / assigned_staff circles

Legacy `album.user` → Circle `author`, `album.assignTo` → Circle
`assigned_staff` (owned_type = collection), via the legacy user id map.
Skip this cell if circles should wait.

In [8]:
conn.rollback()

user_map = {int(k): int(v) for k, v in json.loads(USER_MAP_FILE.read_text()).items()}
print(f"loaded {len(user_map)} user mappings from {USER_MAP_FILE.name}")

author_rows = staff_rows = 0
unmapped_users = []
with conn.cursor() as cur:
    for al in albums:
        if al["id"] not in album_to_collection:
            continue
        a = attrs(al)
        for field, relationship in (("user", "author"), ("assignTo", "assigned_staff")):
            u = rel(a.get(field))
            if not u:
                continue
            new_uid = user_map.get(u["id"])
            if not new_uid:
                unmapped_users.append((al["id"], field, u["id"]))
                continue
            cur.execute(
                """
                INSERT INTO circles (user_id, owned_type, owned_id, relationship)
                VALUES (%s, 'collection', %s, %s)
                ON CONFLICT (user_id, owned_type, owned_id, relationship) DO NOTHING
                """,
                (new_uid, album_to_collection[al["id"]], relationship),
            )
            author_rows += relationship == "author"
            staff_rows += relationship == "assigned_staff"

conn.commit()
print(f"circles upserted: {author_rows} author, {staff_rows} assigned_staff")
print(f"MANUAL REVIEW legacy users not in map ({len(unmapped_users)}): {unmapped_users[:20]}")

AttributeError: 'str' object has no attribute 'read_text'

## 7. Verification

Expected: ~2,768 collections — Properties ~2,108 (2,073 + 35 no-directory
defaults), Restaurants 341, Shopping 235, Events 84; ~2,300 with cover;
~2,790 with country; 1,684+ live; 0 duplicate slugs.

In [9]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT ct.name, COUNT(c.id) FROM collection_types ct
        LEFT JOIN collections c ON c.collection_type_id = ct.id
        GROUP BY ct.id, ct.name ORDER BY MIN(ct.priority)
    """)
    for name, n in cur.fetchall():
        print(f"{name:22}: {n}")
    for label, q in [
        ("collections total",  "SELECT COUNT(*) FROM collections"),
        ("with cover media",   "SELECT COUNT(*) FROM collections WHERE cover_media_id IS NOT NULL"),
        ("with country",       "SELECT COUNT(*) FROM collections WHERE country_id IS NOT NULL"),
        ("with region",        "SELECT COUNT(*) FROM collections WHERE region_id IS NOT NULL"),
        ("with company",       "SELECT COUNT(*) FROM collections WHERE managed_by_company_id IS NOT NULL"),
        ("status = live",      "SELECT COUNT(*) FROM collections WHERE status = 'live'"),
        ("circles",            "SELECT COUNT(*) FROM circles"),
        ("dup slugs (want 0)", "SELECT COUNT(*) FROM (SELECT slug FROM collections GROUP BY slug HAVING COUNT(*) > 1) d"),
    ]:
        cur.execute(q)
        print(f"{label:19}: {cur.fetchone()[0]}")
conn.close()

Properties            : 2108
Restaurants           : 341
Events                : 84
Shopping              : 235
collections total  : 2768
with cover media   : 2365
with country       : 2751
with region        : 2703
with company       : 264
status = live      : 2411
circles            : 0
dup slugs (want 0) : 0
